# Qwen3-Omni

In [1]:
import json
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from transformers import Qwen3OmniMoeForConditionalGeneration, Qwen3OmniMoeProcessor
from qwen_omni_utils import process_mm_info




libgomp: Invalid value for environment variable OMP_NUM_THREADS
/root/miniconda3/envs/qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [2]:
!source /etc/network_turbo

设置成功
注意：仅限于学术用途和加速访问github/huggingface，不承诺稳定性保证


In [3]:
from llm_utils.config import PROJECT_ROOT, CACHE_DIR
MODEL_ID     = "Qwen/Qwen3-Omni-30B-A3B-Instruct"


model = Qwen3OmniMoeForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="flash_attention_2",
    cache_dir=CACHE_DIR,
)
model.disable_talker()

processor = Qwen3OmniMoeProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)

print("Qwen3-Omni model loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_interleaved', 'mrope_section', 'interleaved'}
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section', 'interleaved'}
You are attempting to use Flash Attention 2 without specifying a torch dtype. This might lead to unexpected behaviour
Loading checkpoint shards: 100%|██████████| 15/15 [00:16<00:00,  1.13s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Qwen3-Omni model loaded.


In [4]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [5]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="qwen3_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [6]:
VALID_LABELS = {"Dementia", "Control"}
USE_AUDIO_IN_VIDEO = True


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio": str(wav_path)},
                {"type": "text",  "text": USER_PROMPT},
            ],
        },
    ]

    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=USE_AUDIO_IN_VIDEO)
    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=USE_AUDIO_IN_VIDEO,
    )
    inputs = inputs.to(model.device).to(model.dtype)

    text_ids, _ = model.generate(
        **inputs,
        return_audio=False,
        use_audio_in_video=USE_AUDIO_IN_VIDEO,
        max_new_tokens=64,
    )
    output = processor.batch_decode(
        text_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return output[0]

In [7]:
def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            cleaned = raw.strip().strip("'\".,;:!?").capitalize()
            pred = cleaned if cleaned in VALID_LABELS else None
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [8]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-origin-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt-origin", "Pitt-origin", "Pitt-origin_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt-origin"
evaluate_dataset(csv, audio_dir, "Pitt-origin-raw")

[Pitt-origin-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/Pitt-origin, exists=True


Pitt-origin-raw:   0%|          | 1/552 [00:02<27:31,  3.00s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=002-0 raw='Control' pred=Control


Pitt-origin-raw:   0%|          | 2/552 [00:04<19:09,  2.09s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-origin-raw:   1%|          | 3/552 [00:05<16:30,  1.80s/it]

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Pitt-origin-raw: 100%|██████████| 552/552 [13:00<00:00,  1.41s/it]

[Pitt-origin-raw]
  Accuracy:    0.6014
  F1:          0.7291
  Control Acc: 0.1481
  Dementia Acc:0.9579
  Valid: 552/552  Skipped: 0


In [9]:
# csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
# if not csv.exists() or csv.stat().st_size < 30:
#     create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)

# audio_dir = PROJECT_ROOT / "data/raw/Pitt"
# evaluate_dataset(csv, audio_dir, "Pitt-raw")

In [10]:
csv       = PROJECT_ROOT / "data/processed/ADReSS-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSS", "ADReSS", "ADReSS_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSS"
evaluate_dataset(csv, audio_dir, "ADReSS-raw")

[ADReSS-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/ADReSS, exists=True


ADReSS-raw:   1%|          | 1/156 [00:01<03:24,  1.32s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=S001 raw='Dementia' pred=Dementia


ADReSS-raw:   1%|▏         | 2/156 [00:02<03:19,  1.29s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=S002 raw='Dementia' pred=Dementia


ADReSS-raw:   2%|▏         | 3/156 [00:03<03:13,  1.27s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=S003 raw='Dementia' pred=Dementia


ADReSS-raw: 100%|██████████| 156/156 [03:20<00:00,  1.29s/it]

[ADReSS-raw]
  Accuracy:    0.5192
  F1:          0.6667
  Control Acc: 0.0769
  Dementia Acc:0.9615
  Valid: 156/156  Skipped: 0


In [11]:
csv       = PROJECT_ROOT / "data/processed/ADReSSo-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSSo", "ADReSSo", "ADReSSo_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSSo"
evaluate_dataset(csv, audio_dir, "ADReSSo-raw")

[ADReSSo-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/ADReSSo, exists=True


ADReSSo-raw:   0%|          | 1/237 [00:01<04:57,  1.26s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=adrsdt10 raw='Dementia' pred=Dementia


ADReSSo-raw:   1%|          | 2/237 [00:02<04:59,  1.28s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=adrsdt11 raw='Dementia' pred=Dementia


ADReSSo-raw:   1%|▏         | 3/237 [00:03<04:57,  1.27s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=adrsdt12 raw='Dementia' pred=Dementia


ADReSSo-raw: 100%|██████████| 237/237 [05:07<00:00,  1.30s/it]

[ADReSSo-raw]
  Accuracy:    0.5359
  F1:          0.6857
  Control Acc: 0.0609
  Dementia Acc:0.9836
  Valid: 237/237  Skipped: 0


In [12]:
csv       = PROJECT_ROOT / "data/processed/ADReSS-M-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSS-M", "ADReSS-M", "ADReSS-M_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSS-M"
evaluate_dataset(csv, audio_dir, "ADReSS-M-raw")

[ADReSS-M-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/ADReSS-M, exists=True


ADReSS-M-raw:   0%|          | 1/237 [00:01<05:35,  1.42s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=adrso002 raw='Dementia' pred=Dementia


ADReSS-M-raw:   1%|          | 2/237 [00:02<05:27,  1.40s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=adrso003 raw='Dementia' pred=Dementia


ADReSS-M-raw:   1%|▏         | 3/237 [00:04<05:20,  1.37s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=adrso004 raw='Dementia' pred=Dementia


ADReSS-M-raw: 100%|██████████| 237/237 [05:34<00:00,  1.41s/it]

[ADReSS-M-raw]
  Accuracy:    0.5401
  F1:          0.6804
  Control Acc: 0.1043
  Dementia Acc:0.9508
  Valid: 237/237  Skipped: 0


In [13]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/Lu, exists=True


Lu-raw:   1%|▏         | 1/74 [00:01<01:33,  1.29s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-raw:   3%|▎         | 2/74 [00:02<01:32,  1.28s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-raw:   4%|▍         | 3/74 [00:03<01:30,  1.28s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-raw: 100%|██████████| 74/74 [01:36<00:00,  1.30s/it]

[Lu-raw]
  Accuracy:    0.5135
  F1:          0.6667
  Control Acc: 0.0556
  Dementia Acc:0.9474
  Valid: 74/74  Skipped: 0
